In [0]:
from pyspark.sql import functions as F

def process_silver(batch_df, batch_id):
    # 1. Cast to proper types (bronze is all strings)
    typed = (batch_df
        .withColumn("event_ts",       F.to_timestamp("event_ts"))       # unparseable -> NULL
        .withColumn("coil_temp_c",    F.col("coil_temp_c").cast("double"))
        .withColumn("helium_pct",     F.col("helium_pct").cast("double"))
        .withColumn("vibration_mm_s", F.col("vibration_mm_s").cast("double")))

    # 2. Validation rules -> single reject_reason column
    validated = typed.withColumn("reject_reason",
        F.when(F.col("device_id").isNull(), "null_device_id")
         .when(F.col("event_ts").isNull(), "unparseable_timestamp")
         .when((F.col("coil_temp_c") < 0) | (F.col("coil_temp_c") > 100), "temp_out_of_range")
         .otherwise(None))

    # 3. Quarantine the bad (keep everything -- forensics, reprocessing, audit)
    (validated.filter(F.col("reject_reason").isNotNull())
        .write.mode("append").saveAsTable("fleetpulse.silver.telemetry_quarantine"))

    # 4. Dedupe the good, WITHIN this batch, then MERGE for cross-batch safety
    clean = (validated.filter(F.col("reject_reason").isNull())
        .drop("reject_reason")
        .dropDuplicates(["device_id", "event_ts"]))

    clean.createOrReplaceTempView("batch_clean")
    batch_df.sparkSession.sql("""
        MERGE INTO fleetpulse.silver.telemetry_clean t
        USING batch_clean s
        ON t.device_id = s.device_id AND t.event_ts = s.event_ts
        WHEN NOT MATCHED THEN INSERT *
    """)

# Create the target table first (empty, from schema) -- run once
spark.sql("""
CREATE TABLE IF NOT EXISTS fleetpulse.silver.telemetry_clean (
  device_id STRING, event_ts TIMESTAMP, coil_temp_c DOUBLE, helium_pct DOUBLE,
  vibration_mm_s DOUBLE, error_code STRING, status STRING, firmware_version STRING,
  _ingest_ts TIMESTAMP, _source_file STRING, _rescued_data STRING)
""")

(spark.readStream
    .table("fleetpulse.bronze.telemetry_raw")
    .writeStream
    .foreachBatch(process_silver)
    .option("checkpointLocation", "/Volumes/fleetpulse/raw/checkpoints/silver")
    .trigger(availableNow=True)
    .start())